# miniGPT — Session 3: Context

**Ways of Thinking with Data and Computation · Habib University**

---

On Wednesday you built a die whose faces were words and whose weights were
real word counts. It produced sentences like this:

> *extent day it in well them the the near with way with now only*

The words are plausible. The sentence is not — because every word was chosen
without looking at the words before it.

Today we fix that by giving the model **context**: the word, or two words,
that it has just written.

| Model | Looks at | Data structure |
|---|---|---|
| **unigram** | nothing | `counts[word]` — a dictionary |
| **bigram** | the last word | `counts[w1][w2]` — a dictionary of dictionaries |
| **trigram** | the last two words | `counts[w1][w2][w3]` — three levels deep |

### Two parts, two jobs

- **Part A — the data engine** reads the books, counts, and writes the counts
  to files in `models/`. It never generates a word of text.
- **Part B — miniGPT** loads those files and generates text. It never opens a
  book.

They live in one notebook, but the only thing Part B takes from Part A is the
files on disk.

---
# Part A — The Data Engine

## A0. The tools from Sessions 1 and 2

Run this cell. It is the pipeline you built on Wednesday — reading the books,
stripping the Project Gutenberg boilerplate, cleaning and tokenizing, counting
words — plus your loaded die from Session 1. Nothing in it is new. If you
would rather use your own versions, paste them in instead; just keep the same
function names.

In [2]:
import json
import os
import random

CORPUS_DIR = os.path.join("corpus")
MODEL_DIR = "models"

START_MARKER = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_MARKER = "*** END OF THE PROJECT GUTENBERG EBOOK"
APOSTROPHES = "'\u2019\u2018\u00b4"        # ASCII ', curly \u2019 \u2018, acute \u00b4


def weighted_pick(weights):
    """Pick one key from {outcome: weight}, in proportion to its weight."""
    total = 0
    for outcome in weights:
        total += weights[outcome]

    target = random.random() * total

    running = 0
    for outcome in weights:
        running += weights[outcome]
        if target < running:
            return outcome
    return outcome


def list_corpus_files(folder):
    """Return a sorted list of the .txt file names in `folder`."""
    names = []
    for name in os.listdir(folder):
        if name.endswith(".txt"):
            names.append(name)
    names.sort()
    return names


def read_text_file(path):
    """Return the whole contents of a text file as one string."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def strip_boilerplate(text):
    """Return only the part of `text` between the Gutenberg marker lines."""
    lines = text.split("\n")

    body_lines = []
    inside = False
    found_start = False
    for line in lines:
        if line.startswith(START_MARKER):
            inside = True
            found_start = True
            continue
        if line.startswith(END_MARKER):
            inside = False
            continue
        if inside:
            body_lines.append(line)

    if not found_start:
        print("   WARNING: no start marker found, keeping the whole file")
        return text

    body = ""
    for line in body_lines:
        body += line + "\n"
    return body


def clean_word(word):
    """Normalise one raw token into a comparable word (possibly empty)."""
    result = ""
    for ch in word:
        if ch.isalpha():
            result += ch.lower()
        elif ch in APOSTROPHES:
            result += "'"
    return result.strip("'")


def tokenize(text):
    """Turn a block of text into a list of cleaned words, in order."""
    words = []
    for chunk in text.split():
        for piece in chunk.split("-"):
            word = clean_word(piece)
            if word != "":
                words.append(word)
    return words


def load_corpus(folder):
    """Read every book in `folder`, strip boilerplate, return one token list."""
    all_tokens = []
    for name in list_corpus_files(folder):
        raw = read_text_file(os.path.join(folder, name))
        tokens = tokenize(strip_boilerplate(raw))
        print("   %-14s %8d tokens" % (name, len(tokens)))
        all_tokens += tokens
    return all_tokens


def count_words(tokens):
    """Return {word: how many times it appears} -- the unigram model."""
    counts = {}
    for word in tokens:
        counts[word] = counts.get(word, 0) + 1
    return counts


def join_words(words):
    """Join a list of words into one single-spaced string."""
    out = ""
    for i in range(len(words)):
        if i > 0:
            out += " "
        out += words[i]
    return out


def save_model(counts, filename):
    """Write a counts dictionary to models/filename. Return its size in MB."""
    os.makedirs(MODEL_DIR, exist_ok=True)
    path = os.path.join(MODEL_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(counts, f)
    return os.path.getsize(path) / 1e6


def check(label, got, expected):
    """Compare your answer with the expected one and say which it is."""
    if got == expected:
        print("PASS ", label)
    else:
        print("FAIL ", label)
        print("      expected:", expected)
        print("      got:     ", got)


print("Session 1 and 2 tools loaded.")

Session 1 and 2 tools loaded.


## A1. The unigram model — one level

Exactly Wednesday's model. Read the corpus, count every word, write it out.

```python
unigrams["the"]   ->  how many times "the" appears
```

In [3]:
print("Reading the corpus...")
TOKENS = load_corpus(CORPUS_DIR)
print("total tokens:", len(TOKENS))

UNIGRAMS = count_words(TOKENS)
print()
print("unigram entries:", len(UNIGRAMS))
print("unigrams['the'] =", UNIGRAMS["the"])
print("wrote models/unigram.json (%.1f MB)" % save_model(UNIGRAMS, "unigram.json"))

Reading the corpus...
   pg1342.txt       127853 tokens
   pg2465.txt        24126 tokens
   pg2554.txt       205050 tokens
   pg2641.txt        66885 tokens
   pg2701.txt       214986 tokens
   pg3268.txt       291035 tokens
   pg34413.txt       33826 tokens
   pg59828.txt      442204 tokens
   pg65238.txt       74772 tokens
total tokens: 1480737

unigram entries: 33639
unigrams['the'] = 77683
wrote models/unigram.json (0.5 MB)


---
## A2. The bigram model — two levels

For every word, we want to know **which words came straight after it, and how
often**. That is a dictionary whose values are themselves dictionaries:

```python
bigrams = {
    "the": {"cat": 2, "mat": 1},
    "cat": {"sat": 1, "ran": 1},
    "sat": {"on": 1},
    ...
}
```

```python
bigrams["the"]          ->  {"cat": 2, "mat": 1}   a die for what follows "the"
bigrams["the"]["cat"]   ->  2                      "the cat" appeared twice
```

Read it aloud: *"after **the**, we saw **cat** twice and **mat** once."*
The inner dictionary `bigrams["the"]` has exactly the shape `weighted_pick`
already knows how to roll.

### Trace by hand — on paper, before any code

> *the cat sat on the mat and the cat ran*

Walk along it one pair at a time — (the, cat), (cat, sat), (sat, on), … — and
build the nested dictionary on paper. Check it with your neighbour. You will
check it against the computer in a few minutes.

### Recipe: counting one level down

You already know the counting pattern from `count_words`:

```python
counts[word] = counts.get(word, 0) + 1
```

The nested version is the same thing **one level down**, with one extra step
first:

1. **Make sure the inner dictionary exists.** If `w1` has never been seen
   before, there is no inner dictionary yet — so create an empty one:
   `counts[w1] = {}`.
2. **Count `w2` into the inner dictionary**, with exactly the pattern above.

Run this cell and watch the dictionary grow one pair at a time.

In [14]:
demo = {}
pairs = [("the", "cat"), ("cat", "sat"), ("the", "mat"), ("the", "cat")]

for w1, w2 in pairs:
    # step 1: make sure the inner dictionary exists
    if w1 not in demo:
        demo[w1] = {}
    # step 2: count into it
    inner = demo[w1]
    inner[w2] = inner.get(w2, 0) + 1

    print("after (%s, %s):" % (w1, w2), demo)  # PLEASE DON'T COPY THIS PRINT STATEMENT IN OTHER FUNCTIONS OTHERWISE NOTEBOOK WILL CRASH.

after (the, cat): {'the': {'cat': 1}}
after (cat, sat): {'the': {'cat': 1}, 'cat': {'sat': 1}}
after (the, mat): {'the': {'cat': 1, 'mat': 1}, 'cat': {'sat': 1}}
after (the, cat): {'the': {'cat': 2, 'mat': 1}, 'cat': {'sat': 1}}


Notice that `inner` is not a copy. It is another name for the very same
dictionary that lives inside `demo`, so adding to `inner` changes `demo`.

### Your turn — `add_bigram`

Turn those two steps into a function. It changes `counts` in place and returns
nothing.

In [21]:
def add_bigram(counts, w1, w2):
    """Record one sighting of `w2` straight after `w1`.

    RECIPE (copy the two steps from the demo cell above):
      1. If `w1` is not a key of `counts`, set counts[w1] to {}.
      2. Let `inner` be counts[w1].
      3. Add 1 to inner[w2], using .get(w2, 0) so that a word seen for the
         first time starts from 0.

    Nothing to return -- you are changing `counts` itself.
    """
    # your code here
    
    # step 1: make sure the inner dictionary exists
    if w1 not in counts:
      counts[w1] = {}
    # step 2: count into it
    inner = counts[w1]
    inner[w2] = inner.get(w2, 0) + 1

In [22]:
test = {}
add_bigram(test, "the", "cat")
check("first pair", test, {"the": {"cat": 1}})
add_bigram(test, "the", "cat")
check("same pair again", test, {"the": {"cat": 2}})
add_bigram(test, "the", "mat")
check("new second word", test, {"the": {"cat": 2, "mat": 1}})
add_bigram(test, "cat", "sat")
check("new first word", test, {"the": {"cat": 2, "mat": 1}, "cat": {"sat": 1}})

PASS  first pair
PASS  same pair again
PASS  new second word
PASS  new first word


### Your turn — `build_bigram_counts`

Now walk along the whole token list and call `add_bigram` for every
neighbouring pair.

For the tokens `["the", "cat", "sat", "on"]`:

| `i` | `tokens[i]` (w1) | `tokens[i + 1]` (w2) |
|---|---|---|
| 0 | the | cat |
| 1 | cat | sat |
| 2 | sat | on |

Four tokens, but only **three** pairs: `i` stops at `len(tokens) - 1`,
because the last word has nothing after it. If `i` went one further,
`tokens[i + 1]` would not exist and Python would raise an `IndexError`.

**Recipe**

1. Start with an empty dictionary `counts`.
2. Loop `i` over `range(len(tokens) - 1)`.
3. Call `add_bigram(counts, tokens[i], tokens[i + 1])`.
4. Return `counts`.

In [23]:
def build_bigram_counts(tokens):
    """Return {w1: {w2: count}} -- the bigram model.

    Follow the four-step recipe above. About four lines.
    """
    # your code here
    counts={}
    for i in range(len(tokens)-1):
        add_bigram(counts,tokens[i], tokens[i+1])
    return counts

Check it against the dictionary you built **by hand**.

In [24]:
sentence = tokenize("the cat sat on the mat and the cat ran")
check("bigram counts", build_bigram_counts(sentence), {
    "the": {"cat": 2, "mat": 1},
    "cat": {"sat": 1, "ran": 1},
    "sat": {"on": 1},
    "on":  {"the": 1},
    "mat": {"and": 1},
    "and": {"the": 1},
})

PASS  bigram counts


---
## A3. The trigram model — three levels

Now the model looks at the last **two** words. One more level of nesting:

```python
trigrams["the"]                ->  everything that ever followed "the ..."
trigrams["the"]["cat"]         ->  {"sat": 1, "ran": 1}   a die for what follows "the cat"
trigrams["the"]["cat"]["sat"]  ->  1                      "the cat sat" appeared once
```

For *the cat sat on the mat and the cat ran*, the whole thing is:

```python
{
    "the": {"cat": {"sat": 1, "ran": 1},
            "mat": {"and": 1}},
    "cat": {"sat": {"on": 1}},
    "sat": {"on":  {"the": 1}},
    "on":  {"the": {"mat": 1}},
    "mat": {"and": {"the": 1}},
    "and": {"the": {"cat": 1}},
}
```

### Trace by hand

Take the first four triples — (the, cat, sat), (cat, sat, on), (sat, on, the),
(on, the, mat) — and add them one at a time to an empty dictionary on paper.
At each step, ask: *which levels already exist, and which do I have to
create?*

### Recipe: counting two levels down

Same idea as `add_bigram`, with **one more** "make sure it exists" step:

1. If `w1` is not in `counts`, create `counts[w1] = {}`.
2. If `w2` is not in `counts[w1]`, create `counts[w1][w2] = {}`.
3. Count `w3` into `counts[w1][w2]` with the `.get(w3, 0) + 1` pattern.

The order matters: you cannot look inside `counts[w1]` until step 1 has made
sure it exists.

In [28]:
def add_trigram(counts, w1, w2, w3):
    """Record one sighting of `w3` straight after `w1 w2`.

    RECIPE:
      1. If `w1` is not a key of `counts`, set counts[w1] to {}.
      2. Let `middle` be counts[w1].
      3. If `w2` is not a key of `middle`, set middle[w2] to {}.
      4. Let `inner` be middle[w2].
      5. Add 1 to inner[w3], using .get(w3, 0).

    Look back at add_bigram: steps 3-5 here are add_bigram, one level down.
    """
    # your code here
    if w1 not in counts:
        counts[w1]={}
    middle=counts[w1]
    if w2 not in middle:
        middle[w2]={}
    inner=middle[w2]
    inner[w3] = inner.get(w3, 0) + 1
    

In [29]:
test = {}
add_trigram(test, "the", "cat", "sat")
check("first triple", test, {"the": {"cat": {"sat": 1}}})
add_trigram(test, "the", "cat", "sat")
check("same triple again", test, {"the": {"cat": {"sat": 2}}})
add_trigram(test, "the", "cat", "ran")
check("new third word", test, {"the": {"cat": {"sat": 2, "ran": 1}}})
add_trigram(test, "the", "mat", "and")
check("new second word", test, {"the": {"cat": {"sat": 2, "ran": 1},
                                        "mat": {"and": 1}}})
add_trigram(test, "on", "the", "mat")
check("new first word", test, {"the": {"cat": {"sat": 2, "ran": 1},
                                       "mat": {"and": 1}},
                               "on": {"the": {"mat": 1}}})

PASS  first triple
PASS  same triple again
PASS  new third word
PASS  new second word
PASS  new first word


### Your turn — `build_trigram_counts`

For the tokens `["the", "cat", "sat", "on", "the"]`:

| `i` | `tokens[i]` (w1) | `tokens[i + 1]` (w2) | `tokens[i + 2]` (w3) |
|---|---|---|---|
| 0 | the | cat | sat |
| 1 | cat | sat | on |
| 2 | sat | on | the |

Five tokens, **three** triples: `i` now stops at `len(tokens) - 2`, because
the last two words have no word two places after them.

**Recipe**

1. Start with an empty dictionary `counts`.
2. Loop `i` over `range(len(tokens) - 2)`.
3. Call `add_trigram(counts, tokens[i], tokens[i + 1], tokens[i + 2])`.
4. Return `counts`.

In [30]:
def build_trigram_counts(tokens):
    """Return {w1: {w2: {w3: count}}} -- the trigram model.

    Follow the four-step recipe above. Compare with build_bigram_counts:
    what changes?
    """
    # your code here
    counts = {}
    for i in range(len(tokens)-2):
        add_trigram(counts, tokens[i], tokens[i + 1], tokens[i + 2])
    return counts

In [31]:
check("trigram counts", build_trigram_counts(sentence), {
    "the": {"cat": {"sat": 1, "ran": 1},
            "mat": {"and": 1}},
    "cat": {"sat": {"on": 1}},
    "sat": {"on":  {"the": 1}},
    "on":  {"the": {"mat": 1}},
    "mat": {"and": {"the": 1}},
    "and": {"the": {"cat": 1}},
})

PASS  trigram counts


---
## A4. Build both on the real corpus — and watch the size

### Estimate first

`unigram.json` was about half a megabyte. Before you run the next cell, write
down your guesses: how big will the **bigram** file be? The **trigram**?

In [32]:
BIGRAMS = build_bigram_counts(TOKENS)
different_pairs = 0
for w1 in BIGRAMS:
    different_pairs += len(BIGRAMS[w1])
print("bigram:  %9d different pairs     wrote models/bigram.json  (%.1f MB)"
      % (different_pairs, save_model(BIGRAMS, "bigram.json")))

TRIGRAMS = build_trigram_counts(TOKENS)
different_triples = 0
for w1 in TRIGRAMS:
    for w2 in TRIGRAMS[w1]:
        different_triples += len(TRIGRAMS[w1][w2])
print("trigram: %9d different triples   wrote models/trigram.json (%.1f MB)"
      % (different_triples, save_model(TRIGRAMS, "trigram.json")))

bigram:     440822 different pairs     wrote models/bigram.json  (5.9 MB)
trigram:   1058205 different triples   wrote models/trigram.json (17.7 MB)


Peek inside. These are the real dice the model will roll:

In [33]:
print("words seen after 'the':      ", len(BIGRAMS["the"]))
print("words seen after 'the same': ", len(TRIGRAMS["the"]["same"]))
print("words seen after 'once upon':", TRIGRAMS["once"]["upon"])

words seen after 'the':       9651
words seen after 'the same':  381
words seen after 'once upon': {'raskolnikov': 1, 'a': 2, 'the': 1, 'finding': 1}


### Look at what you just made

- The same books, counted three ways. Nothing was added — no new text, no new
  knowledge — yet every extra word of context made the file **bigger**.
- The vocabulary is about 34,000 words. A table with a slot for **every**
  possible pair would need 34,000 × 34,000 ≈ 1.1 **billion** entries; for
  every triple, 34,000³ ≈ 38 **trillion**. Our nested dictionaries only store
  what actually occurred. That is the only reason this fits on your laptop.
- Open the `models/` folder and look at the file sizes yourself.

That is the end of the data engine. **Part B uses nothing from Part A except
these three files.**

---
# Part B — miniGPT

## B0. Load the models

Imagine this is a different program, on a different computer. It has never
seen a book — only the files in `models/`.

In [34]:
def load_model(filename):
    """Load one counts file from the models/ folder."""
    path = os.path.join(MODEL_DIR, filename)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


unigram_model = load_model("unigram.json")
bigram_model = load_model("bigram.json")
trigram_model = load_model("trigram.json")
print("models loaded.")

models loaded.


## B1. Unigram — no context

Wednesday's model, for comparison. Run it a few times.

In [39]:
def generate_unigram(model, length):
    """Roll the word die `length` times. No context at all."""
    words = []
    for i in range(length):
        words.append(weighted_pick(model))
    return words


print(join_words(generate_unigram(unigram_model, 20)))

inch so the intellect below for and one leave but happy young passion of she with than echoes upon its


---
## B2. Bigram — roll the die for the last word

The generation loop:

> look at the last word → find **its** die → roll it → write the result down →
> repeat.

```
words so far:  ["the"]               last word: "the"   die: bigram_model["the"]   -> rolls "cat"
words so far:  ["the", "cat"]        last word: "cat"   die: bigram_model["cat"]   -> rolls "sat"
words so far:  ["the", "cat", "sat"] ...
```

### Recipe

1. `words = [first_word]`
2. Repeat `length` times:
   1. `last = words[len(words) - 1]` — the most recent word.
   2. If `last` is **not** a key of `model`, the model is stuck: it has never
      seen anything follow that word. Print `"[stuck after '" + last + "']"`
      and return `words` straight away.
   3. Otherwise roll the die: `weighted_pick(model[last])`, and append the
      result to `words`.
3. Return `words`.

In [51]:
def generate_bigram(model, first_word, length):
    """Generate `length` more words, each chosen by the word before it.

    Follow the recipe above.
    """
    # your code here
    words= [first_word]
    for i in range(length):
        last =words[len(words)-1]
        if last not in model:
            print("stuck after "+last)
            return words
        x= weighted_pick(model[last])
        words.append(x)
    return words

### Predict, then run

Better or worse than the unigram? Commit to an answer, then run it several
times. Try other starting words.

In [53]:
print(join_words(generate_bigram(bigram_model, "the", 25)))

the traces hidden and go ben i'd rather past hour mrs oakley to town he was weary where you must be no fins growing agitation paced


---

## B3. Trigram — roll the die for the last two words

Now the die depends on the last **two** words, so the lookup goes two levels
deep: `model[w1][w2]`.

```
words so far:  ["it", "was"]            last two: "it", "was"   die: trigram_model["it"]["was"]  -> rolls "not"
words so far:  ["it", "was", "not"]     last two: "was", "not"  die: trigram_model["was"]["not"] -> rolls "a"
```

### Recipe

1. `words = [w1, w2]`
2. Repeat `length` times:
   1. `a = words[len(words) - 2]` and `b = words[len(words) - 1]` — the last
      two words.
   2. The model is stuck if `a` is not a key of `model`, **or** `b` is not a
      key of `model[a]`. Check them **in that order** — asking for
      `model[a]` when `a` is missing raises a `KeyError`. If stuck, print
      `"[stuck after '" + a + " " + b + "']"` and return `words`.
   3. Otherwise roll the die: `weighted_pick(model[a][b])`, and append the
      result.
3. Return `words`.

In [54]:
def generate_trigram(model, w1, w2, length):
    """Generate `length` more words, each chosen by the two words before it.

    Follow the recipe above. Compare with generate_bigram: what changes?
    """
    # your code here
    words=[w1,w2]
    for i in range(length):
        a = words[len(words)-2]
        b= words[len(words)-1]
        if a not in model or b not in model:
            print("Stuck after",a,b)
            return words
        x=weighted_pick(model[a][b])
        words.append(x)
    return words

In [55]:
print(join_words(generate_trigram(trigram_model, "it", "was", 25)))

it was only a shallow reservoir extending under the shadow always follows choose a wife as father and herself observed enabled her also to be so naughty


### Gallery walk

Run each model a few times and copy your favourite output from each onto the
board, one above the other: unigram, bigram, trigram.